# Resume RAG: retrieval experiments

This notebook ingests the fictional 30-resume corpus, evaluates five job descriptions against labeled expected resumes, and reports top-1 accuracy, mean reciprocal rank, and search latency. The benchmark is illustrative and too small to represent real hiring data.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from resume_rag import ResumeIndex, make_embedder
from job_matcher import match_resumes

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DB_DIR = ROOT / 'chroma_db'
RESUME_DIR = ROOT / 'data' / 'sample_resumes'
embedder = make_embedder('huggingface', 'sentence-transformers/all-MiniLM-L6-v2')
index = ResumeIndex(DB_DIR, 'resume_experiments', embedder)
indexed_count = index.ingest_directory(RESUME_DIR)
print(f'Indexed {indexed_count} resumes; vector count: {index.collection.count()}')

In [ ]:
benchmarks = json.loads((ROOT / 'data' / 'benchmark.json').read_text(encoding='utf-8'))
rows = []
for case in benchmarks:
    job_text = (ROOT / 'data' / 'job_descriptions' / case['job_file']).read_text(encoding='utf-8')
    result = match_resumes(job_text, index, top_k=10)
    expected = case['expected_resume']
    ranked_paths = [candidate['resume_path'] for candidate in result['top_matches']]
    rank = ranked_paths.index(expected) + 1 if expected in ranked_paths else None
    rows.append({
        'job_description': case['job_file'],
        'expected_resume': expected,
        'rank': rank,
        'top_1_correct': rank == 1,
        'reciprocal_rank': 1 / rank if rank else 0.0,
        'latency_ms': result['metrics']['latency_ms'],
        'returned_candidates': len(result['top_matches']),
    })
results = pd.DataFrame(rows)
display(results)
metrics = {
    'retrieval_accuracy_at_1': results['top_1_correct'].mean(),
    'mean_reciprocal_rank_at_10': results['reciprocal_rank'].mean(),
    'mean_latency_ms': results['latency_ms'].mean(),
    'p95_latency_ms': results['latency_ms'].quantile(0.95),
}
metrics

## Interpretation

Accuracy and reciprocal rank compare the expected synthetic resume to the filtered, scored top ten. Latency includes embedding and retrieval but excludes initial model loading and index ingestion. For meaningful quality claims, replace this tiny benchmark with consented, representative data and human-reviewed relevance labels; report results by role and avoid treating ranking scores as calibrated probabilities.